# 02 — Log-uniform mantissa: visual sanity check

Phase 2's derivation rests on one premise: if $X$ is a positive random variable and
$Y = \log_{10}(X) \bmod 1$ is uniform on $[0, 1)$, then the first significant digit of $X$
follows the Benford PMF $P(d) = \log_{10}(1 + 1/d)$.

This notebook does three things:

1. Generate synthetic samples $X = 10^{U}$ with $U \sim \text{Uniform}(0, k)$ for several $k$,
   and visualise how the empirical histogram of $\log_{10}(X) \bmod 1$ approaches
   $\text{Uniform}(0, 1)$ as $k$ grows.
2. Plot the empirical first-digit frequencies side by side with the Benford PMF for the
   widest of those samples.
3. Repeat the log-mantissa check on a real dataset (world city populations) for visual
   confirmation that the premise is realistic.

Output: `figures/log_uniform_intuition.png`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from src.benford import (
    DIGITS,
    benford_pmf,
    empirical_frequencies,
    log_mantissa,
)
from src.datasets import load_world_cities

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FIG_DIR = REPO_ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)

RNG = np.random.default_rng(0)
PMF = benford_pmf()

## 1. Synthetic log-uniform samples

Take $X = 10^{U}$ with $U \sim \text{Uniform}(0, k)$. Then $\log_{10}(X) = U$ is uniform on $[0, k)$, and its fractional part should be uniform on $[0, 1)$ for any positive $k$. Empirically the convergence is visibly cleaner when $k$ spans many orders of magnitude (large sample size helps too).

In [ ]:
K_VALUES = [1, 2, 4, 6]
N = 100_000

samples = {k: 10 ** RNG.uniform(0.0, k, size=N) for k in K_VALUES}
log_mantissas = {k: log_mantissa(x) for k, x in samples.items()}

## 2. The figure

Two rows:
- **Top:** histogram of $Y = \log_{10}(X) \bmod 1$ for each $k$, with the $\text{Uniform}(0, 1)$ density overlaid.
- **Bottom:** empirical first-digit frequencies for the $k = 6$ sample, alongside the same check on world city populations, with the Benford PMF overlaid.

In [ ]:
fig = plt.figure(figsize=(13, 7))
gs = fig.add_gridspec(2, 4, height_ratios=[1, 1.1])

# --- Top row: histograms of log-mantissa for each k ---
for col, k in enumerate(K_VALUES):
    ax = fig.add_subplot(gs[0, col])
    y = log_mantissas[k]
    ax.hist(y, bins=40, range=(0.0, 1.0), density=True, color="#3b6ea8", alpha=0.85)
    ax.axhline(1.0, color="#d68a3e", linestyle="--", linewidth=1.5, label="Uniform(0,1)")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 2.0)
    ax.set_title(f"$k = {k}$\n$X = 10^U,\\ U \\sim \\mathrm{{Unif}}(0, {k})$", fontsize=10)
    ax.set_xlabel("$Y = \\log_{10}(X) \\bmod 1$")
    if col == 0:
        ax.set_ylabel("Density")
        ax.legend(loc="upper right", frameon=False, fontsize=9)
    ax.grid(axis="y", linestyle=":", alpha=0.4)

# --- Bottom row: empirical first-digit vs Benford PMF ---
ax_synth = fig.add_subplot(gs[1, :2])
freq_synth = empirical_frequencies(samples[6])
bar_x = DIGITS - 0.18
line_x = DIGITS + 0.18
ax_synth.bar(bar_x, freq_synth, width=0.36, label="Empirical", color="#3b6ea8")
ax_synth.bar(line_x, PMF, width=0.36, label="Benford", color="#d68a3e")
ax_synth.set_xticks(DIGITS)
ax_synth.set_xlabel("First digit $d$")
ax_synth.set_ylabel("Probability")
ax_synth.set_title(f"Synthetic $X = 10^U,\\ U \\sim \\mathrm{{Unif}}(0, 6)$ ($n = {N:,}$)", fontsize=10)
ax_synth.legend(loc="upper right", frameon=False)
ax_synth.grid(axis="y", linestyle=":", alpha=0.4)

ax_real = fig.add_subplot(gs[1, 2:])
try:
    cities = load_world_cities(min_population=1)
    freq_real = empirical_frequencies(cities["population"].to_numpy())
    n_real = len(cities)
    title_real = f"World city populations ($n = {n_real:,}$)"
except FileNotFoundError as exc:
    freq_real = None
    title_real = f"(skipped: {exc})"

if freq_real is not None:
    ax_real.bar(bar_x, freq_real, width=0.36, label="Empirical", color="#3b6ea8")
    ax_real.bar(line_x, PMF, width=0.36, label="Benford", color="#d68a3e")
    ax_real.legend(loc="upper right", frameon=False)
ax_real.set_xticks(DIGITS)
ax_real.set_xlabel("First digit $d$")
ax_real.set_title(title_real, fontsize=10)
ax_real.grid(axis="y", linestyle=":", alpha=0.4)

fig.suptitle(
    "Log-uniform mantissa $\\Rightarrow$ Benford first-digit distribution",
    fontsize=12,
    y=1.00,
)
fig.tight_layout()
fig.savefig(FIG_DIR / "log_uniform_intuition.png", dpi=300, bbox_inches="tight")
plt.show()

## 3. Reading the figure

- **Top row.** Even at $k = 1$ (one order of magnitude) the histogram of $Y$ is exactly uniform — by construction. For $k \ge 2$ the visual signal is overwhelming: $Y$ fills $[0, 1)$ flatly.
- **Bottom-left.** The empirical first-digit frequencies on the synthetic $k = 6$ sample sit right on top of the Benford PMF. This is the derivation made tangible.
- **Bottom-right.** The same close fit holds on a real dataset that we did not design — world city populations from SimpleMaps. The premise is plausible exactly because the data spans many orders of magnitude.

**Phase 3** repeats this exercise for a different premise — scale invariance under change of unit — and lands on the same Benford curve.